In [1]:
from pathlib import Path
import pandas as pd

# Show numbers with thousand separators and 3 decimals instead of e+09
pd.options.display.float_format = "{:,.3f}".format

# Path to the raw APRA data file
# Notebook runs from /notebooks, so go up one level to the project root
PROJECT_ROOT = Path.cwd().parent
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "Quarterly general insurance performance statistics database September 2023 to June 2026.xlsx"

# 1. Read ONLY the "Database" sheet
df = pd.read_excel(RAW_FILE, sheet_name="Database")

# 2. Print the number of rows and columns
num_rows, num_columns = df.shape
print(f"Number of rows: {num_rows}, Number of columns: {num_columns}")

# 3. Print the first 5 rows
first_5_rows = df.head(5)
print("\nFirst 5 rows:")
print(first_5_rows)

# 4. Print the column names
col_names = df.columns
print("\nColumn names:")
print(col_names)

Number of rows: 29455, Number of columns: 14

First 5 rows:
  Reporting Period                                          Data item  \
0       2026-06-30  Insurance finance expenses for insurance contr...   
1       2026-06-30  Reinsurance finance income for reinsurance con...   
2       2026-06-30  Net insurance financial result through profit ...   
3       2026-06-30                                 Other income items   
4       2026-06-30                                Other expense items   

                     Category                Subject Stock or flow  \
0  Insurance financial result  Financial performance          Flow   
1  Insurance financial result  Financial performance          Flow   
2  Insurance financial result  Financial performance          Flow   
3  Insurance financial result  Financial performance          Flow   
4  Insurance financial result  Financial performance          Flow   

        Industry segment Industry segment group Class of business  \
0  Total di

In [2]:
# How many rows belong to each Subject?
df["Subject"].value_counts()

Subject
Capital adequacy (supplementary)    10259
Class of business performance        5764
Details of investments               3936
Capital adequacy                     3712
Financial performance                2904
Financial position                   2880
Name: count, dtype: int64

In [3]:
# Keep only the class-of-business performance data
cob = df[df["Subject"] == "Class of business performance"]
cob.shape

(5764, 14)

In [4]:
# Which data items are available for each class of business?
cob["Data item"].value_counts()

Data item
Insurance revenue, by class of business and state/territory                                                                          1408
Incurred claims, by class of business and state/territory                                                                            1408
Insurance revenue, by class of business                                                                                               198
Insurance service expense, by class of business                                                                                       198
Insurance service expense -  Incurred claims, by class of business                                                                    198
Insurance service expense - Other insurance service expenses, by class of business                                                    198
Insurance service expense - Amortisation of insurance acquisition cash flows, by class of business                                    198
Insurance service expens

In [5]:
# Insurance revenue by class of business (national level)
revenue = cob[
    (cob["Data item"] == "Insurance revenue, by class of business")
    & (cob["State and territory"].isna())
]
revenue.head()

,Reporting Period,Data item,Category,Subject,Stock or flow,Industry segment,Industry segment group,Class of business,Class of business category,Class of business group,Counterparty grade,State and territory,Stress scenario type,Value
1680,2026-06-30,"Insurance revenue, by class of business",Insurance revenue,Class of business performance,Flow,NaN,NaN,Commercial motor,Short-tail property,Direct insurance,NaN,NaN,NaN,1245000000
1689,2026-06-30,"Insurance revenue, by class of business",Insurance revenue,Class of business performance,Flow,NaN,NaN,Consumer credit,Other,Direct insurance,NaN,NaN,NaN,7000000
1698,2026-06-30,"Insurance revenue, by class of business",Insurance revenue,Class of business performance,Flow,NaN,NaN,CTP,Long-tail,Direct insurance,NaN,NaN,NaN,1049000000
1707,2026-06-30,"Insurance revenue, by class of business",Insurance revenue,Class of business performance,Flow,NaN,NaN,Cyber,Long-tail,Direct insurance,NaN,NaN,NaN,50000000
1716,2026-06-30,"Insurance revenue, by class of business",Insurance revenue,Class of business performance,Flow,NaN,NaN,Directors and officers,Long-tail,Direct insurance,NaN,NaN,NaN,214000000


In [6]:
# Convert value to numeric; masked values ("*") become NaN
cob = cob.copy()
cob["Value"] = pd.to_numeric(cob["Value"], errors="coerce")

# How many values were masked?
cob["Value"].isna().sum()

np.int64(483)

In [7]:
# Map APRA's long names to short column names
ITEMS = {
    "Insurance revenue, by class of business": "insurance_revenue",
    "Insurance service expense -  Incurred claims, by class of business": "incurred_claims",
    "Insurance service expense, by class of business": "insurance_service_expense",
}

national = cob[
    cob["Data item"].isin(ITEMS.keys())
    & cob["State and territory"].isna()
]
national.shape

(594, 14)

In [8]:
# Rotate from vertical to horizontal (pivot)
wide = national.pivot_table(
    index=["Reporting Period", "Class of business"],
    columns ="Data item",
    values ="Value",
    ).rename(columns=ITEMS).reset_index()

wide.columns.name = None
wide.head()

,Reporting Period,Class of business,insurance_revenue,incurred_claims,insurance_service_expense
0,2023-12-31,CTP,"893,000,000.000","-834,000,000.000","-615,000,000.000"
1,2023-12-31,Commercial motor,"1,051,000,000.000","-779,000,000.000","-1,041,000,000.000"
2,2023-12-31,Consumer credit,"10,000,000.000",NaN,"-8,000,000.000"
3,2023-12-31,Cyber,"52,000,000.000","-54,000,000.000","-34,000,000.000"
4,2023-12-31,Directors and officers,"308,000,000.000","-80,000,000.000","-39,000,000.000"


In [9]:
# Loss Ratio and Combined Ratio calculation
wide["loss_ratio"] = -wide["incurred_claims"] / wide["insurance_revenue"]
wide["combined_ratio"] = -wide["insurance_service_expense"] / wide["insurance_revenue"]

In [10]:
# Test calculation for 2026-06-30
wide[
    (wide["Class of business"] == "Householders")
    & (wide["Reporting Period"] == "2026-06-30")
]

,Reporting Period,Class of business,insurance_revenue,incurred_claims,insurance_service_expense,loss_ratio,combined_ratio
188,2026-06-30,Householders,"4,342,000,000.000","-2,094,000,000.000","-2,912,000,000.000",0.482,0.671


In [11]:
# Average combined ratio per class of business across all 11 quarters
summary = (
    wide.groupby("Class of business")["combined_ratio"]
    .mean()
    .sort_values(ascending=False)
)
summary

Class of business
Commercial motor               0.901
CTP                            0.882
Employers liability            0.880
Domestic motor                 0.862
Householders                   0.857
Other direct                   0.843
Travel                         0.835
Public and product liability   0.822
Professional indemnity         0.805
Consumer credit                0.800
Proportional reinsurance       0.785
Marine and aviation            0.769
Fire and ISR                   0.680
Cyber                          0.635
Other accident                 0.632
Directors and officers         0.327
Non-proportional reinsurance   0.232
Mortgage                       0.128
Name: combined_ratio, dtype: float64